## Librerias

In [6]:
import pandas as pd
from pymongo import MongoClient
import json
import os
import configparser
from neo4j import GraphDatabase
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from chromadb.config import Settings
import chromadb
import tensorflow_text
import tensorflow_hub as hub

config = configparser.ConfigParser()
config.read('config.ini')
NEO4J_URI = config.get("NEO4J", "NEO4J_URI")
NEO4J_USER = config.get("NEO4J", "NEO4J_USER")
NEO4J_PASSWORD = config.get("NEO4J", "NEO4J_PASSWORD")
GOOGLE_API_KEY = config.get("GOOGLE", "API_KEY")


## Conexion a LLM

In [40]:
MODELO = 'gemma-4-31b-it' # 'gemini-2.5-flash-lite' #  # 'gemini-2.5-flash' es más rápido pero menos potente que 'gemma-4-31b-it'
llm = ChatGoogleGenerativeAI(
    model=MODELO,
    temperature=0.0,
    thinking_level="minimal",   # Configura el nivel de pensamiento al mínimo
    api_key=GOOGLE_API_KEY
)

def extraer_texto(content):
    """Extrae el texto limpio de la respuesta del modelo"""
    if isinstance(content, str):
        try:
            # Intenta parsear como JSON si es string
            bloques = json.loads(content)
            if isinstance(bloques, list):
                return next((item['text'] for item in bloques if item.get('type') == 'text'), content)
        except:
            pass
    elif isinstance(content, list):
        # Si es lista directamente
        return next((item['text'] for item in content if item.get('type') == 'text'), '')
    
    return str(content)

## Parte 2: Interfaces de Bases de Datos

#### Conectar a las bases de datos: mongoDB, sqlite, Neo4j, chromaDB

In [7]:
# conectar a mongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["reseñas_productos"]
collection_mongo = db["reseñas"]

# Conectar a Base SQLite
import sqlite3
conexion = sqlite3.connect('data/productos.db')

# Conectar a Neo4j
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))


# Cargar Universal Sentence Encoder
embed = hub.load("https://tfhub.dev/google/universal-sentence-encoder-multilingual/3")

# Configuración inicial de ChromaDB
settings = Settings(
    is_persistent=True,
    persist_directory="data/chroma"   # carpeta donde se guardan los datos
)

client = chromadb.Client(settings=settings)
collection_chromadb = client.get_or_create_collection("u5_practica")
print("Número de vectores en la colección:", collection_chromadb.count())

### 2.1. Función de Consulta a Base de Datos

Objetivo: Crear una función para cada tipo de base de datos que ejecute consultas y devuelva resultados.

#### consultar_resenas

In [8]:
def consultar_resenas(filtro: dict) -> list:
    """
    Ejecuta una consulta en la colección de reseñas (MongoDB o JSON).

    Args:
        filtro: Diccionario con criterios de búsqueda
                Ej: {"producto_id": "P001", "puntaje": {"$gte": 4}}
    Returns:
        Lista de reseñas que cumplen el criterio
    """
    
    resultados = collection_mongo.find(filtro)

    return list(resultados)

In [9]:
# Test consulta reseña
filtro_ejemplo = {"producto_id": "P003", "puntaje": {"$gte": 3}}
resenas_filtradas = consultar_resenas(filtro_ejemplo)
print(resenas_filtradas)

[{'_id': ObjectId('6a161babfb7daa06050f562f'), 'fecha': '2024-10-22 08:30', 'usuario': 'Fernando_Morales', 'telefono': '+54 9 11 1234-5678', 'producto_nombre': 'cafetera', 'producto_id': 'P003', 'puntaje': 5, 'comentario': 'Perfecta! Soy muy cafetero y esta cafetera superó mis expectativas. Moler el café fresco hace una diferencia enorme en el sabor. La limpieza es fácil y los controles son intuitivos. Totalmente recomendada para los amantes del café.', 'sentimiento': 'positivo', 'aspectos_positivos': ['sabor', 'limpieza', 'controles'], 'aspectos_negativos': [], 'id_resenia': 'resena_010.txt'}, {'_id': ObjectId('6a161beffb7daa06050f5638'), 'fecha': '2024-09-08 12:15', 'usuario': 'Diana_Figueroa', 'telefono': '+54 9 11 8901-2341', 'producto_nombre': 'cafetera', 'producto_id': 'P003', 'puntaje': 5, 'comentario': 'La cafetera es hermosa y funcional! El café queda delicioso, con mucho aroma. El temporizador es súper práctico. La amo!', 'sentimiento': 'positivo', 'aspectos_positivos': ['est

#### consultar_productos

In [10]:
def consultar_productos(query: str) -> pd.DataFrame:
    """
    Ejecuta una consulta SQL o filtro de pandas.
    Args:
    query (str): Consulta SQL
    Ej: "SELECT * FROM productos WHERE precio_usd < 100"
    Returns:
    pd.DataFrame: DataFrame con los resultados
    """
    # Implementación de la consulta a la base de datos o filtrado de DataFrame
    cursor = conexion.cursor()
    cursor.execute(query)
    filas = cursor.fetchall()
    # convertir a DataFrame
    df = pd.DataFrame(filas, columns=[desc[0] for desc in cursor.description])
    return df


In [11]:
print(consultar_productos("SELECT * FROM productos WHERE precio_usd < 50"))

  id_producto              nombre categoria        marca  precio_usd  stock  \
0        P005     Batidora Manual    Cocina     TechHome       34.99     67   
1        P008           Tostadora    Cocina  ToastMaster       44.99     53   
2        P009  Hervidor Electrico    Cocina   BrewMaster       29.99     89   
3        P014             Waflera    Cocina  ToastMaster       39.99     58   
4        P015       Picadora Mini    Cocina     TechHome       24.99     72   
5        P018     Plancha a Vapor     Hogar   IronMaster       49.99     46   

      color  potencia_w capacidad voltaje  peso_kg  garantia_meses  \
0    Blanco         600        NA    220V      1.1              12   
1     Negro        1000        NA    220V      1.8              12   
2  Plateado        1500      1.7L    220V      1.2              12   
3     Negro         950        NA    220V      2.1              12   
4    Blanco         300      0.5L    220V      0.8              12   
5      Azul        2000   

#### consultar_grafo

In [12]:
def consultar_grafo(query_cypher: str) -> list:
    """Ejecuta una consulta Cypher en la base de grafos (Neo4j)
        Args:
        query_cypher: Consulta en lenguaje Cypher
        Ej: "MATCH (p:Producto {id:'P001'})-[:TIENE_COMPONENTE]->(c) RETURN c.nombre"
        Returns:
        Lista de resultados de la consulta
    """
    # Implementación​
    with driver.session() as session:
        result = session.run(query_cypher)
        return [record for record in result]

In [13]:
# Test consulta grafo
QUERY = "MATCH (p:Producto {id:'P001'})-[:TIENE_COMPONENTE]->(c) RETURN c.nombre"
resultados_grafo = consultar_grafo(QUERY)
print(resultados_grafo)

[<Record c.nombre='Jarra de Vidrio'>, <Record c.nombre='Cuchillas de acero inoxidable 304'>, <Record c.nombre='Motor de Inducción'>]


#### consultar_vectores

In [25]:
def consultar_vectores(query: str, filtro: dict = {}, k: int = 3) -> list:
    """
    Ejecuta una query en la base de datos vectorial
    Args:
    query: Consulta a buscar
    filtro: Diccionario con criterios de búsqueda sobre la metadata
    k: número de vectores devueltos en la respuesta
    Returns:
    Lista de k vectores más cercanos
    """
    embedding_consulta = embed([query]).numpy().tolist()

    results = collection_chromadb.query(
        query_embeddings=embedding_consulta,  # Aquí pasamos el embedding de la consulta
        n_results=k,  # Traemos los k resultados más cercanos
        where=filtro  # Aplicamos el filtro sobre la metadata
    )
    #print(results)
    # imprimir cada resultado pero en formato tabular, utilizando pandas para mostrar los resultados en una tabla, una fila por cada resultado, y las columnas deben ser: IDs, Documento, Distancia y Metadata
    result_ids = results["ids"][0] if results["ids"] else []
    result_docs = results["documents"][0] if results["documents"] else []
    result_distances = results["distances"][0] if results["distances"] else []
    result_metadatas = results["metadatas"][0] if results["metadatas"] else []
    # Devolver una lista de diccionarios con los resultados
    resultados_formateados = []
    for id, doc, dist, meta in zip(result_ids, result_docs, result_distances, result_metadatas):
        resultados_formateados.append({
            "ID": id,
            "Documento": doc,
            "Distancia": dist,
            "Metadata": meta
        })
    return resultados_formateados
    

In [26]:
# Test consulta vectorial
query_vectorial_ejemplo = "Batidora se calienta excesivamente"
filtro_vectorial_ejemplo = {"source_file": "manual_licuadoras.md"}
resultados_vectorial = consultar_vectores(query_vectorial_ejemplo, filtro_vectorial_ejemplo, k=4)
for resultado in resultados_vectorial:
    print(resultado)

{'ID': 'manual_licuadoras.md_fragment_24', 'Documento': '### Batidora se calienta excesivamente\n1. Dejar enfriar 15 minutos entre usos prolongados\n2. Revisar carbones del motor (reemplazar si <5mm)\n3. No usar en velocidad máxima por más de 2 minutos continuos', 'Distancia': 1.2217175960540771, 'Metadata': {'fragment_index': 24, 'source_file': 'manual_licuadoras.md'}}
{'ID': 'manual_licuadoras.md_fragment_3', 'Documento': '#### 1.1 Motor de Inducción\nEl **motor de inducción de 1200W** es el corazón de la Licuadora Roja. Este componente utiliza:\n- Bobinas de cobre de alta conductividad\n- Sistema de ventilación integrado para evitar sobrecalentamiento\n- Protección térmica automática que desconecta el motor a 85°C', 'Distancia': 1.4940993785858154, 'Metadata': {'source_file': 'manual_licuadoras.md', 'fragment_index': 3}}
{'ID': 'manual_licuadoras.md_fragment_23', 'Documento': '---\n\n## Solución de Problemas Comunes\n\n### Licuadora no enciende\n1. Verificar que la jarra esté correc

In [18]:
# Consultar cuantos vectores hay en la colección
def contar_vectores() -> int:
    """
    Cuenta la cantidad de vectores almacenados en la colección de ChromaDB.
    Returns:
        int: Número total de vectores en la colección
    """
    # Implementación
    return collection_chromadb.count()

In [19]:
contador = contar_vectores()
print(f"Cantidad total de vectores en la colección: {contador}")

Cantidad total de vectores en la colección: 561


### 2.2. Función de Información de Base de Datos
Objetivo: Crear funciones que devuelvan metadatos sobre cada base de datos. Esta información se usará para que el LLM construya consultas correctas.

#### info_resenas

In [29]:
def info_resenas() -> dict:
    """
    Devuelve información sobre la estructura de las reseñas.
    """
    total_documentos = collection_mongo.count_documents({})
    campos = collection_mongo.find_one().keys() if total_documentos > 0 else []
    productos_unicos = collection_mongo.distinct("producto_id")
    rango_puntajes = [collection_mongo.find_one(sort=[("puntaje", 1)])["puntaje"], collection_mongo.find_one(sort=[("puntaje", -1)])["puntaje"]]
    ejemplo = collection_mongo.find_one()
    return {
        "total_documentos": total_documentos,
        "campos": list(campos),
        "productos_unicos": productos_unicos,
        "rango_puntajes": rango_puntajes,
        "ejemplo": ejemplo
    }

In [30]:
# Test función info_resenas
info = info_resenas()
print(json.dumps(info, indent=4, default=str))

{
    "total_documentos": 85,
    "campos": [
        "_id",
        "fecha",
        "usuario",
        "telefono",
        "producto_nombre",
        "producto_id",
        "puntaje",
        "comentario",
        "sentimiento",
        "aspectos_positivos",
        "aspectos_negativos",
        "id_resenia"
    ],
    "productos_unicos": [
        null,
        "P001",
        "P002",
        "P003",
        "P004",
        "P005",
        "P006",
        "P007",
        "P008",
        "P009",
        "P010",
        "P011",
        "P012",
        "P013",
        "P014",
        "P015",
        "P016",
        "P017",
        "P018",
        "P019",
        "P020"
    ],
    "rango_puntajes": [
        1,
        5
    ],
    "ejemplo": {
        "_id": "6a1619a8fb7daa06050f55f1",
        "fecha": "2024-10-25 07:45",
        "usuario": "Patricia_Ruiz",
        "telefono": "+54 9 11 8901-2345",
        "producto_nombre": "Cafetera Negra",
        "producto_id": null,
        "punta

#### info_productos

In [33]:
def info_productos() -> dict:
    """
    Devuelve información sobre la tabla de productos (SQL)
    """
    cursor = conexion.cursor()
    cursor.execute("SELECT COUNT(*) FROM productos")
    total_productos = cursor.fetchone()[0]
    cursor.execute("PRAGMA table_info(productos)")
    columnas = [col[1] for col in cursor.fetchall()]
    cursor.execute("SELECT DISTINCT categoria FROM productos")
    categorias = [row[0] for row in cursor.fetchall()]
    cursor.execute("SELECT DISTINCT marca FROM productos")
    marcas = [row[0] for row in cursor.fetchall()]
    cursor.execute("SELECT MIN(precio_usd), MAX(precio_usd) FROM productos")
    rango_precios = cursor.fetchone()
    cursor.execute("SELECT * FROM productos LIMIT 1")
    ejemplo = dict(zip(columnas, cursor.fetchone()))
    return {
        "total_productos": total_productos,
        "columnas": columnas,
        "categorias": categorias,
        "marcas": marcas,
        "rango_precios": rango_precios,
        "ejemplo": ejemplo
    }

In [32]:
# Test función info_productos
info_prod = info_productos()
print(json.dumps(info_prod, indent=4, default=str))

{
    "total_productos": 20,
    "columnas": [
        "id_producto",
        "nombre",
        "categoria",
        "marca",
        "precio_usd",
        "stock",
        "color",
        "potencia_w",
        "capacidad",
        "voltaje",
        "peso_kg",
        "garantia_meses",
        "descripcion"
    ],
    "categorias": [
        "Cocina",
        "Limpieza",
        "Hogar",
        "Climatizacion"
    ],
    "marcas": [
        "TechHome",
        "BrewMaster",
        "SmartClean",
        "BakePro",
        "ToastMaster",
        "HealthyCook",
        "JuicePro",
        "IronMaster",
        "AirFlow"
    ],
    "rango_precios": [
        24.99,
        299.99
    ],
    "ejemplo": {
        "id_producto": "P001",
        "nombre": "Licuadora Roja",
        "categoria": "Cocina",
        "marca": "TechHome",
        "precio_usd": 89.99,
        "stock": 45,
        "color": "Rojo",
        "potencia_w": 1200,
        "capacidad": "2.0L",
        "voltaje": "220V",
 

#### info_grafo

In [36]:
def info_grafo() -> dict:
    """
    Devuelve información sobre el grafo de conocimiento (Neo4j).
    """    
    with driver.session() as session:
        total_nodos = session.run("MATCH (n) RETURN count(n) AS total").single()["total"]
        tipos_nodos = [record["label"] for record in session.run("MATCH (n) UNWIND labels(n) AS label RETURN DISTINCT label")]
        tipos_relaciones = [record["type"] for record in session.run("MATCH ()-[r]->() RETURN DISTINCT type(r) AS type")]
        ejemplo_producto = session.run("MATCH (p:Producto) RETURN p LIMIT 1").single()
        ejemplo_relacion = session.run(
            "MATCH (a)-[r]->(b) RETURN a AS origen, type(r) AS tipo_relacion, b AS destino LIMIT 1"
        ).single()

        ejemplo_relacion_dict = None
        if ejemplo_relacion:
            ejemplo_relacion_dict = {
                "origen": dict(ejemplo_relacion["origen"]) if ejemplo_relacion["origen"] else None,
                "tipo_relacion": ejemplo_relacion["tipo_relacion"],
                "destino": dict(ejemplo_relacion["destino"]) if ejemplo_relacion["destino"] else None,
            }

        return {
            "total_nodos": total_nodos,
            "tipos_nodos": tipos_nodos,
            "tipos_relaciones": tipos_relaciones,
            "ejemplo_producto": dict(ejemplo_producto["p"]) if ejemplo_producto else None,
            "ejemplo_relacion": ejemplo_relacion_dict
        }

In [37]:
# test función info_grafo
info_grafo_result = info_grafo()
print(json.dumps(info_grafo_result, indent=4, default=str))

{
    "total_nodos": 139,
    "tipos_nodos": [
        "Producto",
        "Componente",
        "Procedimiento"
    ],
    "tipos_relaciones": [
        "TIENE_COMPONENTE",
        "TIENE_PROCEDIMIENTO",
        "USADO_EN"
    ],
    "ejemplo_producto": {
        "marca": "HealthyCook / ToastMaster",
        "categoria": "Cocci\u00f3n Saludable sin Aceite",
        "id": "P010",
        "nombre": "Freidora de Aire"
    },
    "ejemplo_relacion": {
        "origen": {
            "marca": "HealthyCook / ToastMaster",
            "categoria": "Cocci\u00f3n Saludable sin Aceite",
            "id": "P010",
            "nombre": "Freidora de Aire"
        },
        "tipo_relacion": "TIENE_COMPONENTE",
        "destino": {
            "tipo": "Resistencia",
            "especificacion": "1400W, Acero inoxidable con recubrimiento cer\u00e1mico",
            "nombre": "Resistencia de Calentamiento"
        }
    }
}


## Parte 3: Generador de Consultas Dinámicas

Crearán un sistema que convierta lenguaje natural en consultas de base de datos usando un LLM.

### 3.1. Generador de Consultas con LLM
Objetivo: Implementar una función que tome una pregunta en lenguaje natural y genere la consulta apropiada para la base de datos correspondiente.

In [52]:
def generar_consulta( pregunta: str, tipo_bd: str, info_estructura: dict) -> str:
    """
    Genera una consulta para una base de datos específica usando un LLM.
    Args:
    pregunta: Pregunta del usuario en lenguaje natural.
    tipo_bd: Tipo de base de datos ("mongodb", "sql", "cypher").
    info_estructura: Información de estructura (de Parte 2.2).

    Returns:
    Consulta generada en el lenguaje apropiado.
    """
    # 1. Construir prompt para el LLM
    prompt_extraction = ChatPromptTemplate.from_messages([
    ("system", "Eres un experto en bases de datos."),
    ("user", """
    Convierte la siguiente pregunta en una consulta {tipo_bd}.

    ESTRUCTURA DE LA BASE DE DATOS:
    {info_estructura}
    
    PREGUNTA DEL USUARIO:
    {pregunta}

    CONSULTA (solo el query, sin codigo adicional ni explicaciones):
    """)
    ])

    chain_extraction = prompt_extraction | llm
    
    response_extraction = chain_extraction.invoke({"tipo_bd": tipo_bd, "info_estructura": json.dumps(info_estructura, indent=4, default=str), "pregunta": pregunta})
   
    consulta = extraer_texto(response_extraction.content)
    
    return consulta

In [54]:
# test generar consulta
pregunta_ejemplo = "Reseñas con 5 estrellas"
tipo_bd_ejemplo = "MongoDB"
info_estructura_ejemplo = info_resenas()  # Puedes usar la función info_resenas() para obtener la estructura de las reseñas
consulta_generada = generar_consulta(pregunta_ejemplo, tipo_bd_ejemplo, info_estructura_ejemplo)
print("Consulta generada por el LLM:")
print(consulta_generada)

Consulta generada por el LLM:
```json
{ "puntaje": 5 }
```


In [55]:
# test generar consulta
pregunta_ejemplo = "Productos de menos de $50"
tipo_bd_ejemplo = "SQL"
info_estructura_ejemplo = info_productos()  # Puedes usar la función info_productos() para obtener la estructura de los productos
consulta_generada = generar_consulta(pregunta_ejemplo, tipo_bd_ejemplo, info_estructura_ejemplo)
print("Consulta generada por el LLM:")
print(consulta_generada)

Consulta generada por el LLM:
```sql
SELECT * FROM productos WHERE precio_usd < 50;
```


In [56]:
# test generar consulta
pregunta_ejemplo = "Componentes de la licuadora"
tipo_bd_ejemplo = "Cypher"
info_estructura_ejemplo = info_grafo()  # Puedes usar la función info_productos() para obtener la estructura de los productos
consulta_generada = generar_consulta(pregunta_ejemplo, tipo_bd_ejemplo, info_estructura_ejemplo)
print("Consulta generada por el LLM:")
print(consulta_generada) # MATCH (p:Producto {nombre:'Licuadora Roja'})-[:TIENE_COMPONENTE]->(c) RETURN c

Consulta generada por el LLM:
MATCH (p:Producto {nombre: "Licuadora"})-[:TIENE_COMPONENTE]->(c:Componente) RETURN c
